[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-04-prompt-engineering.ipynb#scrollTo=da01ea02)

---
# Day 4 · Prompt Engineering Techniques
**certified-journeys / claude-api-certified** · XML, chain-of-thought & few-shot

> **Goal for today:** Master the four techniques that separate good prompts from great ones — XML structure, chain-of-thought, few-shot examples, and role prompting — and test each against your eval set.

In [ ]:
%pip install -q anthropic

In [ ]:
import os
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

def ask(prompt: str, system: str = "", max_tokens: int = 512) -> str:
    kwargs = {"model": MODEL, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[0].text

## Step 1 · XML tags for structure

Claude was trained on documents with XML-like structure. Wrapping prompt sections in named tags makes your intent unambiguous and dramatically reduces misinterpretation.

| Without XML | With XML |
|---|---|
| Claude guesses which part is the instruction | Each section has an explicit label |
| Long context bleeds into instructions | Context is isolated in `<context>` |
| Hard to parse programmatically | Easy to extract `<answer>` blocks |

In [ ]:
context = """The Anthropic Responsible Scaling Policy (RSP) establishes safety commitments
tied to AI capability levels called AI Safety Levels (ASL). ASL-1 is for models posing
minimal risk. ASL-2 is for systems showing early warning signs of dangerous capabilities.
ASL-3 applies to systems that could provide meaningful uplift to those seeking to create
weapons of mass destruction."""

question = "What happens at ASL-3?"

# Without XML
plain_prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"

# With XML
xml_prompt = f"""<context>
{context}
</context>

<question>
{question}
</question>

Answer the question using only information from the context. Put your answer inside <answer> tags."""

plain_response = ask(plain_prompt)
xml_response = ask(xml_prompt)

print("WITHOUT XML:")
print(plain_response[:300])
print("\nWITH XML:")
print(xml_response[:300])

**What just happened?**
- XML tags clarify which text is context, which is the question, and what format the answer should take.
- **`<answer>` tags** let you extract the response programmatically with a simple string split — no regex needed.
- XML works especially well for **multi-document prompts** where context sources need labeling.
- Avoid nesting XML tags more than 2 levels deep — it adds complexity without benefit.

## Step 2 · Chain-of-thought (CoT)

Telling Claude to think before answering improves accuracy on reasoning tasks. The reasoning isn't wasted — it helps Claude catch its own errors before committing to a final answer.

In [ ]:
MATH_PROBLEM = "A store sells apples for $1.50 each and oranges for $2.00 each. If I buy 3 apples and 4 oranges, then return 1 apple, how much do I pay in total?"

# Without CoT
direct = ask(MATH_PROBLEM + " Answer with just the dollar amount.")

# With CoT
cot_prompt = f"""<problem>
{MATH_PROBLEM}
</problem>

Think through this step by step. Show your work in <reasoning> tags, then give the final answer in <answer> tags."""

cot = ask(cot_prompt)

print("Direct answer:")
print(direct)
print("\nChain-of-thought:")
print(cot)

**What just happened?**
- CoT is most valuable for **multi-step problems** — math, logic, legal reasoning, code review.
- The `<reasoning>` block lets you audit whether Claude's logic is sound — not just the final answer.
- **CoT increases token usage** and latency. Don't use it for simple classification tasks.
- For extended reasoning (minutes of thinking), use the extended thinking feature (Day 7).

## Step 3 · Few-shot examples

Few-shot prompting shows Claude 2-5 input/output examples before the real task. It's the most reliable technique for enforcing output format and domain-specific conventions.

In [ ]:
# Task: extract structured data from unstructured text

ZERO_SHOT = """Extract the company name, role, and years of experience from this job posting. Output as JSON.

Posting: {posting}"""

FEW_SHOT = """Extract the company name, role, and years of experience from job postings. Output as JSON.

<examples>
<example>
<posting>Google is hiring a Senior Engineer with 5+ years of Python experience.</posting>
<output>{{"company": "Google", "role": "Senior Engineer", "years_experience": 5}}</output>
</example>
<example>
<posting>Join Stripe as a Product Manager — 3 years minimum in fintech required.</posting>
<output>{{"company": "Stripe", "role": "Product Manager", "years_experience": 3}}</output>
</example>
<example>
<posting>Anthropic seeks a Research Scientist with 7+ years in ML research.</posting>
<output>{{"company": "Anthropic", "role": "Research Scientist", "years_experience": 7}}</output>
</example>
</examples>

Posting: {posting}"""

test_posting = "OpenAI is looking for a Lead Data Engineer with at least 4 years of experience building data pipelines."

r_zero = ask(ZERO_SHOT.format(posting=test_posting))
r_few = ask(FEW_SHOT.format(posting=test_posting))

print("Zero-shot:")
print(r_zero)
print("\nFew-shot:")
print(r_few)

**What just happened?**
- Few-shot examples dramatically improve **format consistency** — Claude learns the exact JSON structure from examples.
- **3 examples is usually enough** for format control. More examples help for nuanced domain tasks.
- Use `{{}}` in Python f-strings to escape literal braces in your JSON examples.
- Ordering examples matters: put the most representative or tricky cases last — Claude pays more attention to recent context.

## Step 4 · Role prompting vs system prompts

Both techniques shape Claude's persona, but they work at different levels:

| Technique | Scope | Best for |
|---|---|---|
| System prompt | Entire conversation | Persistent persona, format rules |
| Role in user message | Single turn | Ad-hoc context, roleplay |
| Combined | Entire conversation with per-turn focus | Complex multi-role scenarios |

In [ ]:
QUESTION = "Should I use a list or a tuple in Python for storing user configuration?"

# System prompt persona
r_system = ask(
    QUESTION,
    system="You are a Python performance engineer at a trading firm. Prioritize speed and memory efficiency over readability. Give direct recommendations backed by benchmarks."
)

# Role in message
r_role = ask(
    f"As a Python educator teaching beginners, answer this question with a simple analogy: {QUESTION}"
)

print("System prompt persona (performance engineer):")
print(r_system[:400])
print("\nRole in message (educator):")
print(r_role[:400])

**What just happened?**
- Same question — radically different answers based on the persona.
- **System prompt personas are stickier** — they persist across turns and are harder for user messages to override.
- **Role in message** is useful for one-off perspective shifts without committing the entire conversation.
- Never rely on role prompting alone for safety-critical behavior — use system prompts + explicit constraints.

## Step 5 · Combine all techniques

Production prompts rarely use a single technique. Here's a template combining all four:

In [ ]:
import json

PRODUCTION_SYSTEM = """You are a senior code reviewer specializing in Python security. Your reviews are terse, actionable, and backed by CVE references where applicable. Always respond in valid JSON."""

def code_review_prompt(code: str) -> str:
    return f"""<examples>
<example>
<code>password = request.args.get('pass')</code>
<output>{{"severity": "HIGH", "issue": "Plaintext password in URL parameter", "recommendation": "Use POST body + HTTPS", "reasoning": "URL parameters logged in server access logs"}}</output>
</example>
<example>
<code>cursor.execute(f"SELECT * FROM users WHERE id={{user_id}}")</code>
<output>{{"severity": "CRITICAL", "issue": "SQL injection via f-string interpolation", "recommendation": "Use parameterized queries", "reasoning": "Direct string interpolation allows arbitrary SQL execution"}}</output>
</example>
</examples>

<code_to_review>
{code}
</code_to_review>

Think briefly about what this code does and any security implications, then output your review as JSON matching the example structure."""


test_code = "os.system(f'ls {user_provided_path}')"

r = client.messages.create(
    model=MODEL,
    max_tokens=256,
    temperature=0,
    system=PRODUCTION_SYSTEM,
    messages=[{"role": "user", "content": code_review_prompt(test_code)}]
)

result_text = r.content[0].text
print("Raw response:", result_text)

try:
    result = json.loads(result_text)
    print(f"\nSeverity: {result['severity']}")
    print(f"Issue: {result['issue']}")
    print(f"Fix: {result['recommendation']}")
except json.JSONDecodeError:
    print("Note: Parse the JSON from the response above")

**What just happened?**
- **System prompt:** sets the persona and output format contract.
- **Few-shot examples:** demonstrate the exact JSON structure.
- **XML tags:** isolate the code under review from the rest of the prompt.
- **CoT hint:** "think briefly" encourages reasoning before the JSON output.
- `temperature=0` ensures consistent, parseable output.

In [ ]:
# Challenge: Build a prompt that extracts action items from meeting notes
# Requirements:
# - Use XML tags to structure the input
# - Use 2 few-shot examples
# - Output: JSON list of {"action": str, "owner": str, "deadline": str}
# - Deadline should be "unspecified" if not mentioned

def extract_action_items(meeting_notes: str) -> list[dict]:
    # Your solution here
    # Hint: build a prompt with examples, call the API, parse JSON
    pass


test_notes = """Meeting 2025-01-15
Alice will send the design mockups by Friday.
Bob needs to update the API documentation.
Team agreed to review the security audit report together next Tuesday.
"""

items = extract_action_items(test_notes)
print(json.dumps(items, indent=2))
assert isinstance(items, list), "Should return a list"
assert len(items) >= 2, "Should extract at least 2 action items"
print("✓ Challenge passed!")

---
## Day 4 key concepts recap

| Technique | When to use | Key benefit |
|---|---|---|
| XML tags | Complex multi-section prompts | Clear boundaries, parseable outputs |
| Chain-of-thought | Math, logic, multi-step reasoning | Catches errors before committing |
| Few-shot examples | Format control, domain conventions | Most reliable format enforcement |
| Role prompting | Persona + perspective control | Changes tone, depth, and framing |

> **Tip:** XML tags are Claude's native language for structured inputs. Always wrap long context, examples, and instructions in named tags.

---
## What's next
**Day 5** → Tool Use & Function Calling: teach Claude to use external APIs, databases, and custom functions.

Mark Day 4 complete in your [tracker](../index.html).